### 1. Kurulum – Notebook’tan src/ modüllerini import etmek ve SparkSession’ı açmak.

In [1]:
import statistics
import sys
import time
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from pyspark.sql import functions as F

from config import SILVER_TRIPS_DIR, SILVER_TRIPS_PARTITIONED_DIR
from spark_session import get_spark

spark = get_spark("nyc-taxi-partitioning")
print(spark.version)

4.3.0


### 2. Partition'lı silver'ı yazmak – Partition’sız silver okunur ve pickup_year / pickup_month ile partition’lı olarak yeniden yazılır. Çıktıda her partition klasörü için dosya sayısı ve toplam boyut görünü

In [2]:
df_silver = spark.read.parquet(str(SILVER_TRIPS_DIR))

df_silver.write.mode("overwrite").partitionBy("pickup_year", "pickup_month").parquet(str(SILVER_TRIPS_PARTITIONED_DIR))

partition_dirs = sorted(SILVER_TRIPS_PARTITIONED_DIR.glob("pickup_year=*/pickup_month=*"))
for partition_dir in partition_dirs:
    parquet_files = list(partition_dir.glob("*.parquet"))
    total_bytes = 0
    for parquet_file in parquet_files:
        total_bytes = total_bytes + parquet_file.stat().st_size
    total_mb = total_bytes / 1024 / 1024
    print(f"{partition_dir.relative_to(SILVER_TRIPS_PARTITIONED_DIR)}: {len(parquet_files)} files, {total_mb:.1f} MB")

pickup_year=2024/pickup_month=1: 14 files, 98.1 MB
pickup_year=2024/pickup_month=2: 14 files, 100.4 MB
pickup_year=2024/pickup_month=3: 14 files, 120.6 MB


### 3. Satır sayısı kontrolü – İki verinin satır sayısı aynı olmalı (9.403.243).

In [3]:
df_non_partitioned = spark.read.parquet(str(SILVER_TRIPS_DIR))
df_partitioned = spark.read.parquet(str(SILVER_TRIPS_PARTITIONED_DIR))

print(f"non_partitioned: {df_non_partitioned.count():,}")
print(f"partitioned:     {df_partitioned.count():,}")

non_partitioned: 9,403,243
partitioned:     9,403,243


### 4. explain() ile partition pruning – Aynı tek ay filtresi (Şubat 2024) iki veride çalışır. Partition’lı planda filtre PartitionFilters içinde görünür: Spark sadece pickup_month=2 klasörünü okur. Partition’sız planda ise filtre PushedFilters içindedir: Spark bütün dosyaları açıp satır satır filtreler. İki sorgunun sonucu aynı olmalı.

In [4]:
month_filter = (F.col("pickup_year") == 2024) & (F.col("pickup_month") == 2)

df_month_non_partitioned = df_non_partitioned.filter(month_filter).agg(F.sum("total_amount").alias("revenue"))
df_month_partitioned = df_partitioned.filter(month_filter).agg(F.sum("total_amount").alias("revenue"))

print("=== non_partitioned ===")
df_month_non_partitioned.explain()
print("=== partitioned ===")
df_month_partitioned.explain()

print(df_month_non_partitioned.collect()[0]["revenue"])
print(df_month_partitioned.collect()[0]["revenue"])

=== non_partitioned ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[sum(total_amount#51)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=120]
      +- HashAggregate(keys=[], functions=[partial_sum(total_amount#51)])
         +- Project [total_amount#51]
            +- Filter (((isnotnull(pickup_year#59) AND isnotnull(pickup_month#60)) AND (pickup_year#59 = 2024)) AND (pickup_month#60 = 2))
               +- FileScan parquet [total_amount#51,pickup_year#59,pickup_month#60] Batched: true, DataFilters: [isnotnull(pickup_year#59), isnotnull(pickup_month#60), (pickup_year#59 = 2024), (pickup_month#60..., Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/silver/yellow_trips], PartitionFilters: [], PushedFilters: [IsNotNull(pickup_year), IsNotNull(pickup_month), EqualTo(pickup_year,2024), EqualTo(pickup_month..., ReadSchema: struct<total_amount:double,pickup_year:int,pickup_month:int>


=== partit

### 5. Süre ölçümü – Aynı sorgu her veride 5 kez çalışır. İlk çalıştırma warm-up’tır (JVM ısınması ve disk cache) ve ayrı gösterilir. Karşılaştırma kalan 4 çalıştırmanın median’ıyla yapılır.

In [5]:
def run_month_query(path: str) -> float:
    df_trips = spark.read.parquet(path)
    df_month = df_trips.filter(month_filter).agg(F.sum("total_amount"))
    start = time.perf_counter()
    df_month.collect()
    return time.perf_counter() - start


scenarios = [
    ("non_partitioned", str(SILVER_TRIPS_DIR)),
    ("partitioned", str(SILVER_TRIPS_PARTITIONED_DIR)),
]

for label, path in scenarios:
    durations = []
    for run in range(5):
        durations.append(run_month_query(path))
    warm_up = durations[0]
    median_seconds = statistics.median(durations[1:])
    print(f"{label}: warm-up {warm_up:.2f} s, median {median_seconds:.2f} s, runs {[round(d, 2) for d in durations]}")

non_partitioned: warm-up 0.29 s, median 0.32 s, runs [0.29, 0.34, 0.36, 0.3, 0.27]
partitioned: warm-up 0.24 s, median 0.23 s, runs [0.24, 0.24, 0.25, 0.22, 0.23]
